# core

> HTMX v4 support for FastHTML

In [ ]:
#| default_exp core

In [ ]:
#| export
import json
from fastcore.meta import delegates
from fasthtml.common import Meta, Script, fast_app, fhjsscr, ft_hx
from fastcore.basics import patch
from fasthtml.core import FastHTML

In [ ]:
#| export
HTMX_V4_SRC = "https://unpkg.com/htmx.org@4.0.0-alpha6/dist/htmx.js"
WS_V4_SRC = "https://unpkg.com/htmx.org@4.0.0-alpha6/dist/ext/hx-ws.js"
DEFAULT_HTMX_V4_CONFIG = {"metaCharacter": "-"}

In [ ]:
#| export
@delegates(ft_hx)
def Partial(*args, **kwargs): return ft_hx("hx-partial")(*args, **kwargs)


In [ ]:
Meta

functools.partial(<function ft_hx at 0x7a7e24cdd260>, 'meta')

In [ ]:
#| export

htmx_v4 = Script(src=HTMX_V4_SRC)
ws_v4 = Script(src=WS_V4_SRC)
meta_cfg = Meta(name="htmx:config", content=json.dumps(DEFAULT_HTMX_V4_CONFIG))
htmx_v4_hdrs = (meta_cfg, fhjsscr, htmx_v4)


In [ ]:
#| export

def fast_app_v4(*args, htmx=False, hdrs=None, **kwargs):
    """`fast_app` with `htmx=False` + v4 header tags added to `hdrs=`."""
    if hdrs is None: hdrs = []
    elif isinstance(hdrs, (list, tuple)): hdrs = list(hdrs)
    else: hdrs = [hdrs]
    hdrs += list(htmx_v4_hdrs)
    return fast_app(*args, htmx=htmx, hdrs=hdrs, **kwargs)

In [ ]:
import fasthtml

FastHTML source code
```
class FastHTML(Starlette):
    def __init__(self, debug=False, routes=None, middleware=None, title: str = "FastHTML page", exception_handlers=None,
                 on_startup=None, on_shutdown=None, lifespan=None, hdrs=None, ftrs=None, exts=None,
                 before=None, after=None, surreal=True, htmx=True, default_hdrs=True, sess_cls=SessionMiddleware,
                 secret_key=None, session_cookie='session_', max_age=365*24*3600, sess_path='/',
                 same_site='lax', sess_https_only=False, sess_domain=None, key_fname='.sesskey',
                 body_wrap=noop_body, htmlkw=None, nb_hdrs=False, canonical=True, **bodykw):
        middleware,before,after = map(_list, (middleware,before,after))
        self.title,self.canonical,self.session_cookie,self.key_fname = title,canonical,session_cookie,key_fname
        hdrs,ftrs,exts = map(listify, (hdrs,ftrs,exts))
        exts = {k:htmx_exts[k] for k in exts}
        htmlkw = htmlkw or {}
        if default_hdrs: hdrs = def_hdrs(htmx, surreal=surreal) + hdrs
        hdrs += [Script(src=ext) for ext in exts.values()]
        if IN_NOTEBOOK:
            hdrs.append(iframe_scr)
            from IPython.display import display,HTML
            if nb_hdrs: display(HTML(to_xml(tuple(hdrs))))
            middleware.append(cors_allow)
        on_startup,on_shutdown = listify(on_startup) or None,listify(on_shutdown) or None
        self.lifespan,self.hdrs,self.ftrs = lifespan,hdrs,ftrs
        self.body_wrap,self.before,self.after,self.htmlkw,self.bodykw = body_wrap,before,after,htmlkw,bodykw
        self.secret_key = get_key(secret_key, key_fname)
        if sess_cls:
            sess = Middleware(sess_cls, secret_key=self.secret_key,session_cookie=session_cookie,
                              max_age=max_age, path=sess_path, same_site=same_site,
                              https_only=sess_https_only, domain=sess_domain)
            middleware.append(sess)
        exception_handlers = ifnone(exception_handlers, {})
        if 404 not in exception_handlers:
            def _not_found(req, exc): return  Response('404 Not Found', status_code=404)
            exception_handlers[404] = _not_found
        excs = {k:_wrap_ex(v, k, hdrs, ftrs, htmlkw, bodykw, body_wrap=body_wrap) for k,v in exception_handlers.items()}
        super().__init__(debug, routes, middleware=middleware, exception_handlers=excs, on_startup=on_startup, on_shutdown=on_shutdown, lifespan=lifespan)
```

I guess now I need to 
- add fasthtml.htmx4src 
- update this `__init__`, adding a bool htmx4 and use the headers of htmx4src in this line `if default_hdrs: hdrs = def_hdrs(htmx, surreal=surreal) + hdrs`
I need to modify this function 
```
def def_hdrs(htmx=True, surreal=True):
    "Default headers for a FastHTML app"
    hdrs = []
    if surreal: hdrs = [surrsrc,scopesrc] + hdrs
    if htmx: hdrs = [htmxsrc,fhjsscr] + hdrs
    return [charset, viewport] + hdrs
```
add htmx4 as args. also check only one htmx or htmx4 is True

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a solid plan. A few things to consider:

1. **For `def_hdrs`** - You'll need to decide what to do when both `htmx=True` and `htmx4=True`. Options:
   - Raise an error
   - Silently prefer one (htmx4?)
   - I'd suggest raising a `ValueError` to make misconfiguration obvious

2. **htmx4 headers** - Looking at your `core` module, you have `htmx_v4_hdrs = (meta_cfg, fhjsscr, htmx_v4)`. Note that `fhjsscr` is shared between both versions - is that intentional? Does FastHTML's JS work unchanged with htmx4?

3. **The meta config** - Your `DEFAULT_HTMX_V4_CONFIG = {"metaCharacter": "-"}` uses `Meta(name="htmx:config", ...)`. This needs to come *before* the htmx script tag in the HTML. Your tuple order looks correct.

Would you like to start with `def_hdrs`? What behavior do you want when both are True?

In [ ]:
htmx4src   = Script(src="https://unpkg.com/htmx.org@4.0.0-alpha6/dist/htmx.js")


In [ ]:
def def_hdrs(htmx=True, htmx4=False, surreal=True):
    "Default headers for a FastHTML app"
    hdrs = []
    if surreal: hdrs = [surrsrc,scopesrc] + hdrs
    if htmx and htmx4: raise ValueError("Cannot enable both htmx and htmx4")
    if htmx: hdrs = [htmxsrc,fhjsscr] + hdrs
    if htmx4: 
        # metaCharacter="-" makes htmx4 use dashes instead of colons (Python-friendly)
        meta_cfg = Meta(name="htmx:config", content=json.dumps({"metaCharacter": "-"}))
        hdrs = [meta_cfg, htmx4src,fhjsscr] + hdrs 
    # TODO: Check if fhjsscr works with htmx4
    return [charset, viewport] + hdrs

In [ ]:
from fasthtml.starlette import *
from fasthtml.core import *
from fastcore.utils import *
from fastcore.xml import *


In [ ]:
# Patch FastHTML.__init__ to add htmx4 support
# - Adds `htmx4=False` parameter to toggle htmx v4 headers
# - Passes htmx4 to def_hdrs() which handles the header selection
@patch
def __init__(self: FastHTML, debug=False, routes=None, middleware=None, title: str = "FastHTML page", exception_handlers=None,
                on_startup=None, on_shutdown=None, lifespan=None, hdrs=None, ftrs=None, exts=None,
                before=None, after=None, surreal=True, htmx=True, htmx4=False, default_hdrs=True, sess_cls=SessionMiddleware,
                secret_key=None, session_cookie='session_', max_age=365*24*3600, sess_path='/',
                same_site='lax', sess_https_only=False, sess_domain=None, key_fname='.sesskey',
                body_wrap=noop_body, htmlkw=None, nb_hdrs=False, canonical=True, **bodykw):
    middleware,before,after = map(_list, (middleware,before,after))
    self.title,self.canonical,self.session_cookie,self.key_fname = title,canonical,session_cookie,key_fname
    hdrs,ftrs,exts = map(listify, (hdrs,ftrs,exts))
    exts = {k:htmx_exts[k] for k in exts}
    htmlkw = htmlkw or {}
    if default_hdrs: hdrs = def_hdrs(htmx, htmx4, surreal=surreal) + hdrs
    hdrs += [Script(src=ext) for ext in exts.values()]
    if IN_NOTEBOOK:
        hdrs.append(iframe_scr)
        from IPython.display import display,HTML
        if nb_hdrs: display(HTML(to_xml(tuple(hdrs))))
        middleware.append(cors_allow)
    on_startup,on_shutdown = listify(on_startup) or None,listify(on_shutdown) or None
    self.lifespan,self.hdrs,self.ftrs = lifespan,hdrs,ftrs
    self.body_wrap,self.before,self.after,self.htmlkw,self.bodykw = body_wrap,before,after,htmlkw,bodykw
    self.secret_key = get_key(secret_key, key_fname)
    if sess_cls:
        sess = Middleware(sess_cls, secret_key=self.secret_key,session_cookie=session_cookie,
                            max_age=max_age, path=sess_path, same_site=same_site,
                            https_only=sess_https_only, domain=sess_domain)
        middleware.append(sess)
    exception_handlers = ifnone(exception_handlers, {})
    if 404 not in exception_handlers:
        def _not_found(req, exc): return  Response('404 Not Found', status_code=404)
        exception_handlers[404] = _not_found
    excs = {k:_wrap_ex(v, k, hdrs, ftrs, htmlkw, bodykw, body_wrap=body_wrap) for k,v in exception_handlers.items()}
    super().__init__(debug, routes, middleware=middleware, exception_handlers=excs, on_startup=on_startup, on_shutdown=on_shutdown, lifespan=lifespan)